# Hydrograph Investigation

Quick notebook for a single example. It loads the observed upstream/downstream gauge hydrographs, lists the available independent and compound RAPID states for that example, and plots:

1. Observed upstream vs downstream gauge hydrographs
2. Observed gauges plus selected RAPID outlet hydrographs

By default the RAPID plot includes all available independent and compound states for the example. Set `SELECTED_STATE_IDS` to a smaller list if needed.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import numpy as np
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import qualitative
from plotly.subplots import make_subplots
from scipy.io import netcdf_file


def find_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "network_variants").exists() and (candidate / "SWORD_gauge_match").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from the current working directory.")


REPO_ROOT = find_repo_root()

# ----------------------------
# notebook configuration
# ----------------------------
EXAMPLE_ID = 21
VARIANT_KIND = "anchored"  # "anchored" or "realized"
INCLUDE_BASE = True
INCLUDE_INDEP = True
INCLUDE_COMPOUND = True
SELECTED_STATE_IDS = None  # None => include all available states for this example
MAX_RAPID_STATES = None    # e.g. 10 to limit the overlay plot

GAUGE_PERIOD_FALLBACK_MODE = "nearest_window"  # "error", "nearest_window", or "full_series"

# Optional manual peak overrides. Leave as None to use the max discharge within the selected gauge window.
UPSTREAM_PEAK_TIME = None
DOWNSTREAM_PEAK_TIME = None

MASK_XLSX = REPO_ROOT / "SWORD_gauge_match/outputs/mask_hydro_quality.xlsx"
TIMESERIES_ROOT = REPO_ROOT / "SWORD_gauge_match/outputs"

example_tag = f"{EXAMPLE_ID:02d}"
print("Repo root:", REPO_ROOT)
print("Example:", example_tag)
print("Variant kind:", VARIANT_KIND)


In [ ]:
def load_quality_row(example_id: int, mask_xlsx: Path) -> pd.Series:
    quality = pd.read_excel(mask_xlsx, sheet_name="Hydrograph_quality")
    row = quality.loc[quality["Example"] == example_id]
    if row.empty:
        raise ValueError(f"No hydrograph-quality row found for Example={example_id}")
    return row.iloc[0]


def candidate_timeseries_paths(station_key: str) -> list[Path]:
    station_key = str(station_key).strip()
    prefix = station_key.split(":", 1)[0] if ":" in station_key else ""
    candidates = []
    if prefix:
        candidates.append(TIMESERIES_ROOT / "subdaily_values" / prefix / "subdaily_timeseries.parquet")
    candidates.extend(
        [
            TIMESERIES_ROOT / "subdaily_timeseries.parquet",
            TIMESERIES_ROOT / "subdaily_timeseries.csv",
            TIMESERIES_ROOT / "gauge_timeseries.parquet",
            TIMESERIES_ROOT / "grdc_timeseries.parquet",
        ]
    )
    seen: set[Path] = set()
    ordered: list[Path] = []
    for path in candidates:
        if path not in seen:
            ordered.append(path)
            seen.add(path)
    return ordered


def load_station_timeseries(station_key: str) -> pd.DataFrame:
    paths = candidate_timeseries_paths(station_key)
    frames: list[pd.DataFrame] = []
    for path in paths:
        if not path.exists():
            continue

        try:
            if path.suffix == ".csv":
                df = pd.read_csv(path)
            elif path.suffix == ".parquet":
                try:
                    df = pd.read_parquet(path, filters=[("station_key", "==", station_key)])
                except Exception:
                    df = pd.read_parquet(path)
            else:
                continue
        except Exception:
            continue

        if "station_key" not in df.columns:
            continue

        df = df.loc[df["station_key"].astype(str) == str(station_key)].copy()
        if df.empty:
            continue

        if "discharge_cms" in df.columns and "discharge" not in df.columns:
            df = df.rename(columns={"discharge_cms": "discharge"})

        if "time" not in df.columns or "discharge" not in df.columns:
            continue

        df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
        df["discharge"] = pd.to_numeric(df["discharge"], errors="coerce")
        df = df.dropna(subset=["time", "discharge"])
        frames.append(df[["station_key", "time", "discharge"]])

    if not frames:
        raise ValueError(f"No timeseries found for station {station_key}")

    return (
        pd.concat(frames, ignore_index=True)
        .sort_values("time")
        .drop_duplicates(subset=["station_key", "time"], keep="first")
        .reset_index(drop=True)
    )


def discover_experiment_dirs(example_tag: str, variant_kind: str) -> dict[str, Path]:
    output_root = REPO_ROOT / "network_variants" / "output"
    directory_names = {
        "indep": f"{example_tag}_indep__{variant_kind}",
        "compound": f"{example_tag}_compound_bubbles__{variant_kind}",
    }

    # Prefer active outputs, but use archived RAPID products when an example
    # (such as the default Example 21) has not been re-run in the active tree.
    roots = [output_root, output_root / "old"]
    found: dict[str, Path] = {}
    for mode, directory_name in directory_names.items():
        for root in roots:
            candidate = root / directory_name
            if candidate.exists():
                found[mode] = candidate
                break
    return found


def build_state_catalog(experiment_dirs: dict[str, Path]) -> pd.DataFrame:
    rows: list[dict] = []
    base_added = False

    for mode, exp_dir in experiment_dirs.items():
        states_dir = exp_dir / "states"
        if not states_dir.exists():
            continue

        for state_dir in sorted(p for p in states_dir.iterdir() if p.is_dir()):
            state_id = state_dir.name
            manifest_path = state_dir / "variant" / "summary" / "variant_manifest.json"
            hydro_path = state_dir / "rapid" / "run" / "outlet_hydrograph.csv"
            if not manifest_path.exists() or not hydro_path.exists():
                continue

            if state_id == "S000_base" and base_added:
                continue

            manifest = json.loads(manifest_path.read_text())
            selected_units = manifest.get("selected_unit_ids", []) or []
            selected_units = [int(x) for x in selected_units]

            if state_id == "S000_base":
                state_kind = "base"
                display_label = "S000_base"
                base_added = True
            elif mode == "compound" or state_id.startswith("CB"):
                state_kind = "compound"
                display_label = f"{state_id} | compound units={selected_units}"
            else:
                state_kind = "unit"
                display_label = f"{state_id} | units={selected_units}"

            rows.append(
                {
                    "mode": mode,
                    "state_kind": state_kind,
                    "state_id": state_id,
                    "selected_unit_ids": selected_units,
                    "selected_unit_ids_text": ", ".join(str(x) for x in selected_units),
                    "n_selected_units": len(selected_units),
                    "display_label": display_label,
                    "state_dir": state_dir,
                    "hydro_path": hydro_path,
                }
            )

    if not rows:
        raise ValueError("No RAPID hydrograph outputs were found for the selected example.")

    catalog = pd.DataFrame(rows)
    kind_order = {"base": 0, "unit": 1, "compound": 2}
    catalog["kind_order"] = catalog["state_kind"].map(kind_order)
    return catalog.sort_values(["kind_order", "mode", "state_id"]).reset_index(drop=True)


def select_states(catalog: pd.DataFrame) -> pd.DataFrame:
    selected = catalog.copy()

    if not INCLUDE_BASE:
        selected = selected.loc[selected["state_kind"] != "base"]
    if not INCLUDE_INDEP:
        selected = selected.loc[selected["mode"] != "indep"]
    if not INCLUDE_COMPOUND:
        selected = selected.loc[selected["mode"] != "compound"]
    if SELECTED_STATE_IDS is not None:
        selected = selected.loc[selected["state_id"].isin(SELECTED_STATE_IDS)]
    if MAX_RAPID_STATES is not None:
        selected = selected.head(MAX_RAPID_STATES)

    return selected.reset_index(drop=True)


def load_outlet_hydrograph(hydro_path: Path) -> pd.DataFrame:
    hydro = pd.read_csv(hydro_path)
    hydro["time_utc"] = pd.to_datetime(hydro["time_utc"], utc=True, errors="coerce")
    hydro["q_outlet_cms"] = pd.to_numeric(hydro["q_outlet_cms"], errors="coerce")
    hydro = hydro.dropna(subset=["time_utc", "q_outlet_cms"]).copy()
    return hydro.sort_values("time_utc").reset_index(drop=True)


def subset_timeseries_to_period(
    frame: pd.DataFrame,
    period_start: pd.Timestamp,
    period_end: pd.Timestamp,
    station_key: str,
    fallback_mode: str = "error",
) -> tuple[pd.DataFrame, str]:
    subset = frame.loc[frame["time"].between(period_start, period_end)].reset_index(drop=True)
    if not subset.empty:
        label = f"requested period [{period_start}, {period_end}]"
        return subset, label

    available_start = frame["time"].min() if not frame.empty else pd.NaT
    available_end = frame["time"].max() if not frame.empty else pd.NaT
    message = (
        f"No gauge observations found for station {station_key} in the requested period "
        f"[{period_start}, {period_end}]. Available range is [{available_start}, {available_end}]."
    )
    if frame.empty:
        raise ValueError(message)

    if fallback_mode == "full_series":
        print("Warning:", message)
        print("Using the full available series for this station instead.")
        label = f"full available series [{available_start}, {available_end}]"
        return frame.reset_index(drop=True), label

    if fallback_mode == "nearest_window":
        duration = period_end - period_start
        midpoint = period_start + duration / 2
        nearest_idx = (frame["time"] - midpoint).abs().idxmin()
        nearest_time = pd.Timestamp(frame.loc[nearest_idx, "time"])
        window_start = nearest_time - duration / 2
        window_end = nearest_time + duration / 2
        fallback_subset = frame.loc[frame["time"].between(window_start, window_end)].reset_index(drop=True)
        if not fallback_subset.empty:
            print("Warning:", message)
            print(
                f"Using nearest available gauge window for {station_key}: "
                f"[{window_start}, {window_end}] centered on {nearest_time}."
            )
            label = f"nearest available window [{window_start}, {window_end}]"
            return fallback_subset, label

    if fallback_mode != "error":
        raise ValueError(
            f"Unsupported fallback_mode={fallback_mode!r}. Use 'error', 'nearest_window', or 'full_series'."
        )

    raise ValueError(message)


def choose_peak_time(frame: pd.DataFrame, manual_time=None, series_name: str = "series") -> pd.Timestamp:
    if frame.empty:
        raise ValueError(f"Cannot choose a peak time from an empty {series_name} series.")

    if manual_time is None:
        idx = frame["discharge"].idxmax()
        return pd.Timestamp(frame.loc[idx, "time"])

    target = pd.Timestamp(manual_time)
    if target.tzinfo is None:
        target = target.tz_localize("UTC")
    else:
        target = target.tz_convert("UTC")

    nearest_idx = (frame["time"] - target).abs().idxmin()
    return pd.Timestamp(frame.loc[nearest_idx, "time"])


def add_peak_line(
    fig: go.Figure,
    peak_time: pd.Timestamp,
    line_color: str,
    *,
    row: int | None = None,
    col: int | None = None,
    line_dash: str = "dot",
    opacity: float = 0.5,
) -> None:
    kwargs = {
        "x": peak_time,
        "line_dash": line_dash,
        "line_color": line_color,
        "opacity": opacity,
    }
    if row is not None and col is not None:
        fig.add_vline(row=row, col=col, **kwargs)
    else:
        fig.add_vline(**kwargs)


def add_peak_lines(fig: go.Figure, upstream_peak: pd.Timestamp, downstream_peak: pd.Timestamp) -> None:
    fig.add_vline(x=upstream_peak, line_dash="dash", line_color="#1f77b4", opacity=0.9)
    fig.add_vline(x=downstream_peak, line_dash="dash", line_color="#d62728", opacity=0.9)
    fig.add_annotation(x=upstream_peak, y=1.0, yref="paper", text="Up peak", showarrow=False, yshift=10, font=dict(color="#1f77b4"))
    fig.add_annotation(x=downstream_peak, y=1.0, yref="paper", text="Down peak", showarrow=False, yshift=10, font=dict(color="#d62728"))


def print_example_lists(catalog: pd.DataFrame) -> None:
    indep = catalog.loc[catalog["mode"] == "indep", ["state_id", "selected_unit_ids_text"]]
    compound = catalog.loc[catalog["mode"] == "compound", ["state_id", "selected_unit_ids_text"]]

    print("Independent states:")
    if indep.empty:
        print("  none")
    else:
        for _, row in indep.iterrows():
            print(f"  {row['state_id']}: [{row['selected_unit_ids_text']}]")

    print("\nCompound states:")
    if compound.empty:
        print("  none")
    else:
        for _, row in compound.iterrows():
            print(f"  {row['state_id']}: [{row['selected_unit_ids_text']}]")


In [ ]:
quality_row = load_quality_row(EXAMPLE_ID, MASK_XLSX)
up_station = quality_row["Up"]
down_station = quality_row["Down"]
period_start = pd.to_datetime(quality_row["Period_start"], utc=True)
period_end = pd.to_datetime(quality_row["Period_end"], utc=True)

experiment_dirs = discover_experiment_dirs(example_tag, VARIANT_KIND)
catalog = build_state_catalog(experiment_dirs)


In [ ]:

# catalog1 = catalog.iloc[[0, 23, 14, 30]]
catalog1 = catalog
selected_catalog = select_states(catalog1)

up_ts = load_station_timeseries(up_station)
down_ts = load_station_timeseries(down_station)
up_ts, up_window_label = subset_timeseries_to_period(
    up_ts,
    period_start,
    period_end,
    str(up_station),
    fallback_mode=GAUGE_PERIOD_FALLBACK_MODE,
)
down_ts, down_window_label = subset_timeseries_to_period(
    down_ts,
    period_start,
    period_end,
    str(down_station),
    fallback_mode=GAUGE_PERIOD_FALLBACK_MODE,
)

up_peak_time = choose_peak_time(up_ts, UPSTREAM_PEAK_TIME, series_name=f"upstream gauge {up_station}")
down_peak_time = choose_peak_time(down_ts, DOWNSTREAM_PEAK_TIME, series_name=f"downstream gauge {down_station}")

print("Up station:", up_station)
print("Down station:", down_station)
print("Period:", period_start, "to", period_end)
print("Up gauge window:", up_window_label)
print("Down gauge window:", down_window_label)
print("Detected experiment dirs:")
for mode, exp_dir in experiment_dirs.items():
    print(f"  {mode}: {exp_dir}")
print("\nSelected upstream peak:", up_peak_time)
print("Selected downstream peak:", down_peak_time)
print("\nAvailable state catalog:")
display(catalog[["mode", "state_kind", "state_id", "selected_unit_ids_text", "display_label"]])
print_example_lists(catalog)

if selected_catalog.empty:
    raise ValueError("The current selection produced no RAPID states to plot.")

rapid_series = []
for row in selected_catalog.itertuples(index=False):
    hydro = load_outlet_hydrograph(Path(row.hydro_path))
    rapid_series.append(
        {
            "mode": row.mode,
            "state_kind": row.state_kind,
            "state_id": row.state_id,
            "display_label": row.display_label,
            "hydro": hydro,
        }
    )

selected_catalog[["mode", "state_kind", "state_id", "selected_unit_ids_text", "display_label"]]


In [ ]:
fig_obs = go.Figure()
fig_obs.add_trace(
    go.Scatter(
        x=up_ts["time"],
        y=up_ts["discharge"],
        mode="lines",
        name=f"Observed upstream: {up_station}",
        line=dict(color="#1f77b4", width=3),
    )
)
fig_obs.add_trace(
    go.Scatter(
        x=down_ts["time"],
        y=down_ts["discharge"],
        mode="lines",
        name=f"Observed downstream: {down_station}",
        line=dict(color="#d62728", width=3),
    )
)
add_peak_lines(fig_obs, up_peak_time, down_peak_time)
fig_obs.update_layout(
    title=f"Example {example_tag}: observed upstream vs downstream gauges",
    xaxis_title="Time (UTC)",
    yaxis_title="Discharge (cms)",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_obs.show()

fig_rapid = go.Figure()
fig_rapid.add_trace(
    go.Scatter(
        x=up_ts["time"],
        y=up_ts["discharge"],
        mode="lines",
        name=f"Observed upstream: {up_station}",
        line=dict(color="#1f77b4", width=3),
    )
)
fig_rapid.add_trace(
    go.Scatter(
        x=down_ts["time"],
        y=down_ts["discharge"],
        mode="lines",
        name=f"Observed downstream: {down_station}",
        line=dict(color="#d62728", width=3),
    )
)

rapid_palette = qualitative.Dark24 + qualitative.Alphabet + qualitative.Set3
for i, item in enumerate(rapid_series):
    hydro = item["hydro"]
    if item["state_kind"] == "base":
        color = "#111111"
        width = 3
        opacity = 1.0
    else:
        color = rapid_palette[i % len(rapid_palette)]
        width = 1.8
        opacity = 0.75

    fig_rapid.add_trace(
        go.Scatter(
            x=hydro["time_utc"],
            y=hydro["q_outlet_cms"],
            mode="lines",
            name=item["display_label"],
            line=dict(color=color, width=width),
            opacity=opacity,
        )
    )
    rapid_peak_time = hydro.loc[hydro["q_outlet_cms"].idxmax(), "time_utc"]
    fig_rapid.add_vline(x=rapid_peak_time, line_dash="dot", line_color=color, opacity=0.35)

add_peak_lines(fig_rapid, up_peak_time, down_peak_time)
fig_rapid.update_layout(
    title=f"Example {example_tag}: observed gauges + RAPID outlet hydrographs",
    xaxis_title="Time (UTC)",
    yaxis_title="Discharge (cms)",
    template="plotly_white",
    height=700,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
)
fig_rapid.show()


In [ ]:
# Observed gauges and RAPID outlet hydrographs for the paper results figure.
# Run the configuration and data-loading cells above before this cell.
fig_obs = go.Figure()
fig_obs.add_trace(
    go.Scatter(
        x=up_ts["time"],
        y=up_ts["discharge"],
        mode="lines",
        name=f"Observed upstream: {up_station}",
        line=dict(color="#1f77b4", width=3),
    )
)
fig_obs.add_trace(
    go.Scatter(
        x=down_ts["time"],
        y=down_ts["discharge"],
        mode="lines",
        name=f"Observed downstream: {down_station}",
        line=dict(color="#d62728", width=3),
    )
)
add_peak_lines(fig_obs, up_peak_time, down_peak_time)
fig_obs.update_layout(
    title=f"Example {example_tag}: observed upstream vs downstream gauges",
    xaxis_title="Time (UTC)",
    yaxis_title="Discharge (cms)",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_obs.show()

fig_rapid = go.Figure()
fig_rapid.add_trace(
    go.Scatter(
        x=up_ts["time"],
        y=up_ts["discharge"],
        mode="lines",
        name=f"Observed upstream: {up_station}",
        line=dict(color="#1f77b4", width=3),
    )
)
fig_rapid.add_trace(
    go.Scatter(
        x=down_ts["time"],
        y=down_ts["discharge"],
        mode="lines",
        name=f"Observed downstream: {down_station}",
        line=dict(color="#d62728", width=3),
    )
)

rapid_palette = qualitative.Dark24 + qualitative.Alphabet + qualitative.Set3
for i, item in enumerate(rapid_series):
    hydro = item["hydro"]
    if item["state_kind"] == "base":
        color = "#111111"
        width = 3
        opacity = 1.0
    else:
        color = rapid_palette[i % len(rapid_palette)]
        width = 1.8
        opacity = 0.75

    fig_rapid.add_trace(
        go.Scatter(
            x=hydro["time_utc"],
            y=hydro["q_outlet_cms"],
            mode="lines",
            name=item["display_label"],
            line=dict(color=color, width=width),
            opacity=opacity,
        )
    )
    rapid_peak_time = hydro.loc[hydro["q_outlet_cms"].idxmax(), "time_utc"]
    fig_rapid.add_vline(x=rapid_peak_time, line_dash="dot", line_color=color, opacity=0.35)

add_peak_lines(fig_rapid, up_peak_time, down_peak_time)
fig_rapid.update_layout(
    title=f"Example {example_tag}: observed gauges + RAPID outlet hydrographs",
    xaxis_title="Time (UTC)",
    yaxis_title="Discharge (cms)",
    template="plotly_white",
    height=700,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
)
fig_rapid.show()


In [ ]:
# Local Hydrographs

# Local hydrographs

In [ ]:
# ----------------------------
# local downstream pulse diagnostic
# ----------------------------
DIAG_STATE_ID = "CB002_bubble_10"
DIAG_MODE = "compound"  # "indep" or "compound"
DIAG_TARGET_PARENT_LINK_ID = 11
DIAG_N_DOWNSTREAM = 3
DIAG_NORMALIZE_REACH_PEAKS = False


def resolve_qout_path(run_dir: Path) -> Path:
    candidates = [
        run_dir / "Qout_rapid_framework.nc",
        run_dir / "qout.nc",
    ]
    for path in candidates:
        if path.exists():
            return path
    nc_files = sorted(run_dir.glob("*.nc"))
    if len(nc_files) == 1:
        return nc_files[0]
    raise FileNotFoundError(f"No Qout NetCDF file found in {run_dir}")


def load_qout_stack(run_dir: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    qout_path = resolve_qout_path(run_dir)
    with netcdf_file(qout_path, "r", mmap=False) as ds:
        river_ids = np.array(ds.variables["rivid"].data, dtype=np.int64).copy()
        time_seconds = np.array(ds.variables["time"].data, dtype=np.float64).copy()
        qout = np.array(ds.variables["Qout"].data, dtype=np.float64).copy()
    return river_ids, time_seconds, qout


def get_state_row(catalog_frame: pd.DataFrame, state_id: str) -> pd.Series:
    row = catalog_frame.loc[catalog_frame["state_id"] == state_id]
    if row.empty:
        raise ValueError(f"State {state_id!r} not found in catalog.")
    return row.iloc[0]


def load_state_paths(state_row: pd.Series) -> dict[str, Path]:
    hydro_path = Path(state_row["hydro_path"])
    state_dir = hydro_path.parents[2]
    return {
        "state_dir": state_dir,
        "run_dir": hydro_path.parent,
        "prep_dir": state_dir / "rapid" / "prep",
        "matching_dir": state_dir / "variant" / "matching",
    }


def load_rapid_links(prep_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(prep_dir / "rapid_link_attributes.csv")
    for col in ["reach_id", "parent_link_id", "subreach_index", "subreach_count", "id_us_node", "id_ds_node"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def terminal_reach_for_parent(rapid_links: pd.DataFrame, parent_link_id: int) -> pd.Series:
    local = rapid_links.loc[pd.to_numeric(rapid_links["parent_link_id"], errors="coerce").eq(int(parent_link_id))].copy()
    if local.empty:
        raise ValueError(f"No rapid subreaches found for parent_link_id={parent_link_id}.")
    if "subreach_index" in local.columns and local["subreach_index"].notna().any():
        local = local.sort_values(["subreach_index", "reach_id"], ascending=[True, True], kind="mergesort")
        return local.iloc[-1]
    local = local.sort_values(["reach_id"], kind="mergesort")
    return local.iloc[-1]


def build_parent_downstream_path(rapid_links: pd.DataFrame, start_parent_link_id: int, n_downstream: int) -> list[int]:
    parent_terminal = {}
    parent_initial = {}
    for parent_link_id, group in rapid_links.groupby(pd.to_numeric(rapid_links["parent_link_id"], errors="coerce"), sort=False):
        if pd.isna(parent_link_id):
            continue
        g = group.copy()
        if "subreach_index" in g.columns and g["subreach_index"].notna().any():
            g = g.sort_values(["subreach_index", "reach_id"], kind="mergesort")
        else:
            g = g.sort_values(["reach_id"], kind="mergesort")
        parent_initial[int(parent_link_id)] = g.iloc[0]
        parent_terminal[int(parent_link_id)] = g.iloc[-1]

    us_lookup: dict[int, list[int]] = {}
    for parent_link_id, row in parent_initial.items():
        us_lookup.setdefault(int(row["id_us_node"]), []).append(parent_link_id)

    path = [int(start_parent_link_id)]
    current = int(start_parent_link_id)
    for _ in range(int(n_downstream)):
        terminal = parent_terminal[current]
        next_candidates = [pid for pid in us_lookup.get(int(terminal["id_ds_node"]), []) if pid != current]
        if not next_candidates:
            break
        current = sorted(next_candidates)[0]
        path.append(int(current))
    return path


def parse_parent_link_list(value: object) -> list[int]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    return [int(part.strip()) for part in text.split(",") if str(part).strip()]


def parent_links_for_unit_ids(base_links: pd.DataFrame, unit_ids: list[int]) -> list[int]:
    wanted = {int(value) for value in unit_ids}
    if not wanted:
        return []

    def row_has_unit(value: object) -> bool:
        return bool(wanted.intersection(parse_parent_link_list(value)))

    mask = pd.Series(False, index=base_links.index)
    for col in ["unit_ids", "root_unit_ids", "collapsed_unit_ids"]:
        if col in base_links.columns:
            mask = mask | base_links[col].apply(row_has_unit)

    if "parent_link_id" not in base_links.columns:
        return []
    parent_ids = pd.to_numeric(base_links.loc[mask, "parent_link_id"], errors="coerce").dropna().astype(int)
    return sorted(parent_ids.unique().tolist())


def resolve_base_link_mapping(matching_dir: Path, child_parent_link_id: int, base_links: pd.DataFrame) -> dict[str, object]:
    lineage = pd.read_csv(matching_dir / "link_lineage.csv")
    row = lineage.loc[pd.to_numeric(lineage["id_link"], errors="coerce").eq(int(child_parent_link_id))]
    if row.empty:
        raise ValueError(f"No lineage row found for child link {child_parent_link_id}.")
    row = row.iloc[0]

    collapsed_unit_ids = parse_parent_link_list(row.get("collapsed_unit_ids", ""))
    if collapsed_unit_ids:
        parent_link_ids = parent_links_for_unit_ids(base_links, collapsed_unit_ids)
        if parent_link_ids:
            exit_node = pd.to_numeric(pd.Series([row.get("matched_parent_ds_node")]), errors="coerce").iloc[0]
            return {
                "base_parent_link_ids": parent_link_ids,
                "base_exit_node": int(exit_node) if pd.notna(exit_node) else None,
                "comparison_mode": "collapsed_unit_set",
            }

    matched = parse_parent_link_list(row.get("matched_parent_link_ids", ""))
    if not matched:
        raise ValueError(f"No matched parent links recorded for child link {child_parent_link_id}.")
    terminal = int(matched[-1])
    terminal_row = terminal_reach_for_parent(base_links, terminal)
    return {
        "base_parent_link_ids": [terminal],
        "base_exit_node": int(terminal_row["id_ds_node"]),
        "comparison_mode": "single_parent",
    }


def build_downstream_path_from_us_node(rapid_links: pd.DataFrame, start_us_node: int, n_downstream: int) -> list[int]:
    parent_terminal = {}
    parent_initial = {}
    for parent_link_id, group in rapid_links.groupby(pd.to_numeric(rapid_links["parent_link_id"], errors="coerce"), sort=False):
        if pd.isna(parent_link_id):
            continue
        g = group.copy()
        if "subreach_index" in g.columns and g["subreach_index"].notna().any():
            g = g.sort_values(["subreach_index", "reach_id"], kind="mergesort")
        else:
            g = g.sort_values(["reach_id"], kind="mergesort")
        parent_initial[int(parent_link_id)] = g.iloc[0]
        parent_terminal[int(parent_link_id)] = g.iloc[-1]

    us_lookup: dict[int, list[int]] = {}
    for parent_link_id, row in parent_initial.items():
        us_lookup.setdefault(int(row["id_us_node"]), []).append(parent_link_id)

    start_candidates = sorted(us_lookup.get(int(start_us_node), []))
    if not start_candidates:
        return []

    path = [int(start_candidates[0])]
    current = int(start_candidates[0])
    for _ in range(max(int(n_downstream) - 1, 0)):
        terminal = parent_terminal[current]
        next_candidates = [pid for pid in us_lookup.get(int(terminal["id_ds_node"]), []) if pid != current]
        if not next_candidates:
            break
        current = sorted(next_candidates)[0]
        path.append(int(current))
    return path


def load_reach_series(run_dir: Path, prep_dir: Path, reach_id: int) -> pd.DataFrame:
    river_ids, time_seconds, qout = load_qout_stack(run_dir)
    index_by_reach = {int(rid): idx for idx, rid in enumerate(river_ids)}
    qout_path = resolve_qout_path(run_dir)
    if int(reach_id) not in index_by_reach:
        raise ValueError(f"Reach {reach_id} not found in {qout_path}")
    outlet = pd.read_csv(run_dir / "outlet_hydrograph.csv")
    outlet["time_utc"] = pd.to_datetime(outlet["time_utc"], utc=True, errors="coerce")
    series = pd.DataFrame(
        {
            "time_seconds": time_seconds,
            "time_utc": outlet["time_utc"].to_numpy(),
            "q_cms": qout[:, index_by_reach[int(reach_id)]],
        }
    )
    return series


def collect_downstream_reach_pairs(
    catalog_frame: pd.DataFrame,
    *,
    variant_state_id: str,
    variant_parent_link_id: int,
    n_downstream: int,
) -> pd.DataFrame:
    variant_row = get_state_row(catalog_frame, variant_state_id)
    base_row = get_state_row(catalog_frame, "S000_base")
    variant_paths = load_state_paths(variant_row)
    base_paths = load_state_paths(base_row)
    variant_links = load_rapid_links(variant_paths["prep_dir"])
    base_links = load_rapid_links(base_paths["prep_dir"])

    variant_parent_path = build_parent_downstream_path(variant_links, variant_parent_link_id, n_downstream)
    base_mapping = resolve_base_link_mapping(variant_paths["matching_dir"], variant_parent_link_id, base_links)
    base_parent_ids_step0 = list(base_mapping["base_parent_link_ids"])
    base_exit_node = base_mapping["base_exit_node"]
    base_downstream_path = build_downstream_path_from_us_node(base_links, int(base_exit_node), n_downstream) if base_exit_node is not None else []

    rows = []
    for idx, variant_parent in enumerate(variant_parent_path):
        variant_terminal = terminal_reach_for_parent(variant_links, variant_parent)
        if idx == 0:
            base_terminals = [terminal_reach_for_parent(base_links, parent_id) for parent_id in base_parent_ids_step0]
            rows.append(
                {
                    "step": idx,
                    "variant_parent_link_id": int(variant_parent),
                    "variant_reach_id": int(variant_terminal["reach_id"]),
                    "base_parent_link_ids": ",".join(str(int(parent_id)) for parent_id in base_parent_ids_step0),
                    "base_reach_ids": ",".join(str(int(row["reach_id"])) for row in base_terminals),
                    "base_parent_count": len(base_parent_ids_step0),
                }
            )
            continue
        if (idx - 1) >= len(base_downstream_path):
            break
        base_parent = base_downstream_path[idx - 1]
        base_terminal = terminal_reach_for_parent(base_links, base_parent)
        rows.append(
            {
                "step": idx,
                "variant_parent_link_id": int(variant_parent),
                "variant_reach_id": int(variant_terminal["reach_id"]),
                "base_parent_link_ids": str(int(base_parent)),
                "base_reach_ids": str(int(base_terminal["reach_id"])),
                "base_parent_count": 1,
            }
        )
    return pd.DataFrame(rows)


def plot_local_downstream_comparison(
    catalog_frame: pd.DataFrame,
    *,
    variant_state_id: str,
    variant_parent_link_id: int,
    n_downstream: int,
    normalize: bool = False,
):
    pairs = collect_downstream_reach_pairs(
        catalog_frame,
        variant_state_id=variant_state_id,
        variant_parent_link_id=variant_parent_link_id,
        n_downstream=n_downstream,
    )
    if pairs.empty:
        raise ValueError("No downstream reach pairs could be resolved.")

    variant_row = get_state_row(catalog_frame, variant_state_id)
    base_row = get_state_row(catalog_frame, "S000_base")
    variant_paths = load_state_paths(variant_row)
    base_paths = load_state_paths(base_row)

    fig = make_subplots(
        rows=len(pairs),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=[
            (
                "Terminal reach of changed object"
                if int(row.step) == 0
                else f"{int(row.step)} reach{'es' if int(row.step) > 1 else ''} downstream"
            )
            + f" | base parent(s) {row.base_parent_link_ids} vs variant parent {int(row.variant_parent_link_id)}"
            for row in pairs.itertuples(index=False)
        ],
    )

    summary_rows = []
    for row_idx, pair in enumerate(pairs.itertuples(index=False), start=1):
        base_parent_ids = [int(value) for value in str(pair.base_parent_link_ids).split(",") if str(value).strip()]
        base_reach_ids = [int(value) for value in str(pair.base_reach_ids).split(",") if str(value).strip()]
        base_series_list = [
            load_reach_series(base_paths["run_dir"], base_paths["prep_dir"], reach_id).copy()
            for reach_id in base_reach_ids
        ]
        variant_series = load_reach_series(variant_paths["run_dir"], variant_paths["prep_dir"], int(pair.variant_reach_id)).copy()

        base_peak = max(float(series["q_cms"].max()) for series in base_series_list)
        variant_peak = float(variant_series["q_cms"].max())
        if normalize and base_peak > 0 and variant_peak > 0:
            for series in base_series_list:
                series["plot_q"] = series["q_cms"] / base_peak
            variant_series["plot_q"] = variant_series["q_cms"] / variant_peak
            y_title = "Normalized discharge [-]"
        else:
            for series in base_series_list:
                series["plot_q"] = series["q_cms"]
            variant_series["plot_q"] = variant_series["q_cms"]
            y_title = "Discharge (cms)"

        for base_idx, base_series in enumerate(base_series_list):
            base_name = (
                f"Base parent {base_parent_ids[base_idx]}"
                if len(base_parent_ids) > 1
                else "Base"
            )
            fig.add_trace(
                go.Scatter(
                    x=base_series["time_utc"],
                    y=base_series["plot_q"],
                    mode="lines",
                    name=base_name if row_idx == 1 else None,
                    legendgroup=f"base_{base_idx}",
                    showlegend=row_idx == 1,
                    line=dict(color="#111111", width=2.5 if len(base_series_list) == 1 else 1.4),
                    opacity=1.0 if len(base_series_list) == 1 else 0.7,
                ),
                row=row_idx,
                col=1,
            )
            base_peak_time_single = base_series.loc[base_series["q_cms"].idxmax(), "time_utc"]
            add_peak_line(fig, base_peak_time_single, "#111111", row=row_idx, col=1, line_dash="dot", opacity=0.35)
        
        fig.add_trace(
            go.Scatter(
                x=variant_series["time_utc"],
                y=variant_series["plot_q"],
                mode="lines",
                name=variant_state_id if row_idx == 1 else None,
                legendgroup="variant",
                showlegend=row_idx == 1,
                line=dict(color="#d62728", width=2.0),
            ),
            row=row_idx,
            col=1,
        )
        variant_peak_time = variant_series.loc[variant_series["q_cms"].idxmax(), "time_utc"]
        add_peak_line(fig, variant_peak_time, "#d62728", row=row_idx, col=1, line_dash="dash", opacity=0.45)

        base_peak_times = [series.loc[series["q_cms"].idxmax(), "time_utc"] for series in base_series_list]
        base_peak_time = min(base_peak_times)
        summary_rows.append(
            {
                "step": int(pair.step),
                "base_parent_link_ids": pair.base_parent_link_ids,
                "variant_parent_link_id": int(pair.variant_parent_link_id),
                "base_reach_ids": pair.base_reach_ids,
                "variant_reach_id": int(pair.variant_reach_id),
                "base_peak_time_utc": base_peak_time,
                "variant_peak_time_utc": variant_peak_time,
                "variant_minus_base_peak_minutes": (variant_peak_time - base_peak_time).total_seconds() / 60.0,
            }
        )

    fig.update_yaxes(title_text=y_title, row=1, col=1)
    fig.update_xaxes(title_text="Time (UTC)", row=len(pairs), col=1)
    fig.update_layout(
        title=f"Example {example_tag}: local downstream hydrograph tracking for {variant_state_id}",
        template="plotly_white",
        height=max(350 * len(pairs), 500),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig.show()

    display(pd.DataFrame(summary_rows))
    return pairs


local_pairs = plot_local_downstream_comparison(
    catalog,
    variant_state_id=DIAG_STATE_ID,
    variant_parent_link_id=DIAG_TARGET_PARENT_LINK_ID,
    n_downstream=DIAG_N_DOWNSTREAM,
    normalize=DIAG_NORMALIZE_REACH_PEAKS,
)
local_pairs


# Paper Results

In [ ]:
# ----------------------------
# Paper Results — functions
# ----------------------------
# This cell can run in a fresh kernel; it does not rely on the exploratory cells above.
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import qualitative

import matplotlib.pyplot as plt
import numpy as np

def plateau_mask(t, y, window=40, max_rel_range=0.005, max_rel_slope=1e-4):
    """Mark samples that belong to sufficiently flat discharge windows."""
    t_hours = (pd.to_datetime(t) - pd.to_datetime(t).iloc[0]).dt.total_seconds() / 3600
    t_values, y_values = np.asarray(t_hours), np.asarray(y)
    stable = np.zeros(len(y_values), dtype=bool)
    for index in range(len(y_values) - window + 1):
        tt, yy = t_values[index : index + window], y_values[index : index + window]
        level = np.median(yy)
        if level == 0:
            continue
        q05, q95 = np.percentile(yy, [5, 95])
        rel_range = (q95 - q05) / abs(level)
        rel_slope = abs(np.polyfit(tt, yy, 1)[0]) / abs(level)
        if rel_range < max_rel_range and rel_slope < max_rel_slope:
            stable[index : index + window] = True
    return stable


def paper_find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "network_variants").exists() and (candidate / "SWORD_gauge_match").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current working directory.")


PAPER_REPO_ROOT = paper_find_repo_root()
PAPER_MASK_XLSX = PAPER_REPO_ROOT / "SWORD_gauge_match/outputs/mask_hydro_quality.xlsx"
PAPER_TIMESERIES_ROOT = PAPER_REPO_ROOT / "SWORD_gauge_match/outputs"


def paper_load_quality_row(example_id: int) -> pd.Series:
    quality = pd.read_excel(PAPER_MASK_XLSX, sheet_name="Hydrograph_quality")
    rows = quality.loc[quality["Example"] == example_id]
    if rows.empty:
        raise ValueError(f"No hydrograph-quality row found for Example={example_id}.")
    return rows.iloc[0]


def paper_load_station_timeseries(station_key: str) -> pd.DataFrame:
    station_key = str(station_key).strip()
    prefix = station_key.split(":", 1)[0] if ":" in station_key else ""
    paths = []
    if prefix:
        paths.append(PAPER_TIMESERIES_ROOT / "subdaily_values" / prefix / "subdaily_timeseries.parquet")
    paths.extend([
        PAPER_TIMESERIES_ROOT / "subdaily_timeseries.parquet",
        PAPER_TIMESERIES_ROOT / "subdaily_timeseries.csv",
        PAPER_TIMESERIES_ROOT / "gauge_timeseries.parquet",
        PAPER_TIMESERIES_ROOT / "grdc_timeseries.parquet",
    ])
    frames = []
    for path in dict.fromkeys(paths):
        if not path.exists():
            continue
        try:
            frame = pd.read_csv(path, low_memory=False) if path.suffix == ".csv" else pd.read_parquet(path)
        except Exception:
            continue
        if "station_key" not in frame.columns:
            continue
        frame = frame.loc[frame["station_key"].astype(str) == station_key].copy()
        if frame.empty:
            continue
        if "discharge_cms" in frame.columns and "discharge" not in frame.columns:
            frame = frame.rename(columns={"discharge_cms": "discharge"})
        if {"time", "discharge"}.issubset(frame.columns):
            frame["time"] = pd.to_datetime(frame["time"], utc=True, errors="coerce")
            frame["discharge"] = pd.to_numeric(frame["discharge"], errors="coerce")
            frames.append(frame[["station_key", "time", "discharge"]].dropna())
    if not frames:
        raise ValueError(f"No timeseries found for station {station_key}.")
    return pd.concat(frames, ignore_index=True).sort_values("time").drop_duplicates(
        subset=["station_key", "time"], keep="first"
    ).reset_index(drop=True)


def paper_subset_to_period(frame, start, end, station_key, fallback_mode):
    subset = frame.loc[frame["time"].between(start, end)].reset_index(drop=True)
    if not subset.empty:
        return subset
    if fallback_mode == "full_series":
        return frame.reset_index(drop=True)
    if fallback_mode == "nearest_window" and not frame.empty:
        midpoint = start + (end - start) / 2
        nearest_time = frame.loc[(frame["time"] - midpoint).abs().idxmin(), "time"]
        window = frame.loc[frame["time"].between(nearest_time - (end - start) / 2, nearest_time + (end - start) / 2)]
        if not window.empty:
            return window.reset_index(drop=True)
    raise ValueError(f"No observations for {station_key} in the requested period.")


def paper_peak_time(frame, manual_time):
    if manual_time is None:
        return pd.Timestamp(frame.loc[frame["discharge"].idxmax(), "time"])
    target = pd.Timestamp(manual_time)
    target = target.tz_localize("UTC") if target.tzinfo is None else target.tz_convert("UTC")
    return pd.Timestamp(frame.loc[(frame["time"] - target).abs().idxmin(), "time"])


def paper_terminal_plateau_start(frame, time_column, value_column, minimum_constant_tail_samples=2, relative_tolerance=0.005):
    """Find a near-constant terminal plateau, allowing small numerical wriggles."""
    frame = frame.reset_index(drop=True)
    values = frame[value_column].to_numpy(dtype=float)
    reference_samples = min(len(values), max(20, minimum_constant_tail_samples))
    terminal_level = np.median(values[-reference_samples:])
    tolerance = max(abs(terminal_level) * relative_tolerance, 1e-9)
    outside_plateau = np.flatnonzero(np.abs(values - terminal_level) > tolerance)
    tail_start_position = int(outside_plateau[-1]) + 1 if len(outside_plateau) else 0
    tail_length = len(frame) - tail_start_position
    return (
        frame[time_column].iloc[tail_start_position]
        if tail_length >= minimum_constant_tail_samples
        else frame[time_column].iloc[-1]
    )


def paper_figure_xlim(observed_frames, rapid_hydrographs):
    """End when every selected RAPID run has receded to observed post-peak baseflow."""
    start = min(frame["time"].min() for frame in observed_frames)
    observed_baseflow = min(
        frame.loc[frame.index >= frame["discharge"].idxmax(), "discharge"].min()
        for frame in observed_frames
    )
    receding_times = []
    for hydro in rapid_hydrographs:
        peak_position = hydro["q_outlet_cms"].to_numpy().argmax()
        recession = hydro.iloc[peak_position:]
        at_baseflow = recession.loc[recession["q_outlet_cms"] <= observed_baseflow]
        receding_times.append(
            at_baseflow["time_utc"].iloc[0]
            if not at_baseflow.empty
            else paper_terminal_plateau_start(hydro, "time_utc", "q_outlet_cms")
        )
    return start, max(receding_times)


def paper_figure_ylim(observed_frames, rapid_hydrographs, xlim, margin_fraction=0.03):
    values = []
    for frame in observed_frames:
        values.extend(frame.loc[frame["time"].between(*xlim), "discharge"].tolist())
    for hydro in rapid_hydrographs:
        values.extend(hydro.loc[hydro["time_utc"].between(*xlim), "q_outlet_cms"].tolist())
    minimum, maximum = min(values), max(values)
    padding = max((maximum - minimum) * margin_fraction, 1e-6)
    return minimum - padding, maximum + padding


def paper_add_observed_peaks(fig, upstream_peak, downstream_peak):
    for peak, color in [(upstream_peak, "#1f77b4"), (downstream_peak, "#d62728")]:
        fig.add_vline(x=peak, line_dash="dash", line_color=color, opacity=0.9)


def paper_build_catalog(example_tag: str, variant_kind: str) -> pd.DataFrame:
    output_root = PAPER_REPO_ROOT / "network_variants/output"
    rows = []
    base_added = False
    for mode, name in {
        "indep": f"{example_tag}_indep__{variant_kind}",
        "compound": f"{example_tag}_compound_bubbles__{variant_kind}",
    }.items():
        experiment_dir = next((root / name for root in [output_root, output_root / "old"] if (root / name).exists()), None)
        if experiment_dir is None:
            continue
        bubble_summary_path = experiment_dir / "states/S000_base/hierarchy/bubble_summary.csv"
        bubble_complexity = {}
        if bubble_summary_path.exists():
            bubble_summary = pd.read_csv(bubble_summary_path)
            bubble_complexity = dict(zip(bubble_summary["compound_bubble_id"], bubble_summary["n_compound_units"]))
        for state_dir in sorted((experiment_dir / "states").glob("*")):
            if not state_dir.is_dir():
                continue
            manifest_path = state_dir / "variant/summary/variant_manifest.json"
            hydro_path = state_dir / "rapid/run/outlet_hydrograph.csv"
            if not manifest_path.exists() or not hydro_path.exists():
                continue
            state_id = state_dir.name
            if state_id == "S000_base" and base_added:
                continue
            selected_unit_ids = [int(unit) for unit in json.loads(manifest_path.read_text()).get("selected_unit_ids", [])]
            state_kind = "base" if state_id == "S000_base" else ("compound" if mode == "compound" or state_id.startswith("CB") else "unit")
            if state_kind == "base":
                label = "Base network"
            elif state_kind == "unit":
                label = f"Simple unit {selected_unit_ids[0]}"
            else:
                bubble_id = json.loads(manifest_path.read_text()).get("compound_bubble_id")
                compound_type = "Nested compound" if bubble_complexity.get(bubble_id, 0) > 0 else "Interconnected compound"
                label = f"{compound_type}: units {', '.join(map(str, selected_unit_ids))}"
            rows.append({
                "mode": mode, "state_kind": state_kind, "state_id": state_id,
                "selected_unit_ids": selected_unit_ids,
                "selected_unit_ids_text": ", ".join(map(str, selected_unit_ids)),
                "display_label": label, "hydro_path": hydro_path,
            })
            base_added = base_added or state_kind == "base"
    if not rows:
        raise ValueError(f"No RAPID outlet hydrographs found for Example {example_tag}.")
    catalog = pd.DataFrame(rows)
    return catalog.sort_values("state_id").reset_index(drop=True)


def paper_load_outlet_hydrograph(hydro_path: Path) -> pd.DataFrame:
    hydro = pd.read_csv(hydro_path)
    hydro["time_utc"] = pd.to_datetime(hydro["time_utc"], utc=True, errors="coerce")
    hydro["q_outlet_cms"] = pd.to_numeric(hydro["q_outlet_cms"], errors="coerce")
    return hydro.dropna(subset=["time_utc", "q_outlet_cms"]).sort_values("time_utc").reset_index(drop=True)


# Paper Results API
def paper_retrieve_data(settings):
    """Load all available RAPID states for one example/variant."""
    defaults = {"example_id": 21, "variant_kind": "anchored", "state_mode": "both", "include_base": True, 
                "unit_or_state_selection": None, "show_vertical_peak_lines": False, "y_margin_fraction": 0.03,
                  "gauge_period_fallback_mode": "nearest_window"}
    config = {**defaults, **settings}
    tag = f"{config['example_id']:02d}"
    quality = paper_load_quality_row(config["example_id"])
    up_station, down_station = quality["Up"], quality["Down"]
    start, end = pd.to_datetime(quality["Period_start"], utc=True), pd.to_datetime(quality["Period_end"], utc=True)
    catalog = paper_build_catalog(tag, config["variant_kind"])
    up = paper_subset_to_period(paper_load_station_timeseries(up_station), start, end, str(up_station), config["gauge_period_fallback_mode"])
    down = paper_subset_to_period(paper_load_station_timeseries(down_station), start, end, str(down_station), config["gauge_period_fallback_mode"])
    rapid = [{"mode": row.mode, "state_kind": row.state_kind, "state_id": row.state_id,
               "selected_unit_ids": row.selected_unit_ids, "display_label": row.display_label, 
               "hydro": paper_load_outlet_hydrograph(Path(row.hydro_path))} for row in catalog.itertuples(index=False)]
    data = {"config": config, "example_tag": tag, "up_ts": up, "down_ts": down, "up_peak": paper_peak_time(up, None), 
            "down_peak": paper_peak_time(down, None), "catalog": catalog, "rapid": rapid}
    print(f"Retrieved {len(rapid)} RAPID states for Example {tag}.")
    return data


def paper_build_figures(data, settings=None):
    """Filter loaded states and create figures without re-reading any data."""
    config = {**data["config"], **(settings or {})}
    modes = {"independent": {"indep"}, "compound": {"compound"}, "both": {"indep", "compound"}}
    # Enrich data cached by the earlier retrieval-cell version, which did not
    # retain catalog metadata in each RAPID-series record.
    rapid_records = data["rapid"]
    if rapid_records and "mode" not in rapid_records[0]:
        rapid_records = [
            {
                **item,
                "mode": row.mode,
                "state_id": row.state_id,
                "selected_unit_ids": row.selected_unit_ids,
            }
            for item, row in zip(rapid_records, data["catalog"].itertuples(index=False))
        ]
    series = [item for item in rapid_records if item["mode"] in modes[config["state_mode"]] and (config["include_base"] 
                                                                                                 or item["state_kind"] != "base")]
    if config["unit_or_state_selection"] is not None:
        selection = set(config["unit_or_state_selection"])
        series = [item for item in series if (config["include_base"] and item["state_kind"] == "base") or item["state_id"] 
                  in selection or bool(set(item["selected_unit_ids"]).intersection(selection))]
    if not series: raise ValueError("No loaded RAPID states match the plotting selection.")
    xlim = paper_figure_xlim([data["up_ts"], data["down_ts"]], [item["hydro"] for item in series])
    ylim = paper_figure_ylim([data["up_ts"], data["down_ts"]], [item["hydro"] for item in series], xlim, config["y_margin_fraction"])
    observed = go.Figure()
    observed.add_trace(go.Scatter(x=data["up_ts"]["time"], y=data["up_ts"]["discharge"], mode="lines", 
                                  name="Observed upstream", line=dict(color="#1f77b4", width=3)))
    observed.add_trace(go.Scatter(x=data["down_ts"]["time"], y=data["down_ts"]["discharge"], mode="lines", 
                                  name="Observed downstream", line=dict(color="#d62728", width=3)))
    if config["show_vertical_peak_lines"]: paper_add_observed_peaks(observed, data["up_peak"], data["down_peak"])
    observed.update_layout( xaxis_title="Time (UTC)", yaxis_title="Discharge (cms)", template="plotly_white", height=500, 
                           legend=dict(orientation="h", yanchor="bottom", y=1.02, 
                                       xanchor="left", x=0)); observed.update_xaxes(range=xlim); observed.update_yaxes(range=ylim)
    rapid_fig = go.Figure(observed); palette = qualitative.Dark24 + qualitative.Alphabet + qualitative.Set3
    for i,item in enumerate(sorted(series, key=lambda item: item["state_kind"] == "base")):
        hydro, base = item["hydro"], item["state_kind"] == "base"; 
        color = "#111111" if base else palette[i % len(palette)]
        rapid_fig.add_trace(go.Scatter(x=hydro["time_utc"], y=hydro["q_outlet_cms"], 
                                       mode="lines", name=item["display_label"], 
                                       line=dict(color=color, width=3.5 if base else 1.8), opacity=1 if base else .75))
        if config["show_vertical_peak_lines"]: rapid_fig.add_vline(x=hydro.loc[hydro["q_outlet_cms"].idxmax(), "time_utc"], 
                                                                   line_dash="dot", line_color=color, opacity=.35)
    # rapid_fig.update_layout(title=f"Example {data['example_tag']}: observed gauges + RAPID outlet hydrographs", xaxis_title="Time 
    # (UTC)", yaxis_title="Discharge (cms)", template="plotly_white", height=700, legend=dict(orientation="v", yanchor="top", y=1, 
    # xanchor="left", x=1.02)); rapid_fig.update_xaxes(range=xlim); rapid_fig.update_yaxes(range=ylim)
    rapid_fig.update_layout(xaxis_title="Time (UTC)", yaxis_title="Discharge (cms)", template="plotly_white", height=700, 
                            legend=dict(orientation="v", yanchor="top", y=1, 
                                        xanchor="left", x=1.02)); rapid_fig.update_xaxes(range=xlim);rapid_fig.update_yaxes(range=ylim)
    return {"observed_gauges": observed, "rapid_overlay": rapid_fig}


def paper_build_matplotlib_simple_figure(data, settings=None):
    """Create an unshifted Matplotlib RAPID-overlay figure."""
    config = {**data["config"], **(settings or {})}
    modes = {"independent": {"indep"}, "compound": {"compound"}, "both": {"indep", "compound"}}
    records = data["rapid"]
    if records and "mode" not in records[0]:
        records = [{**item, "mode": row.mode, "state_id": row.state_id, "selected_unit_ids": row.selected_unit_ids} for item, row in zip(records, data["catalog"].itertuples(index=False))]
    series = [item for item in records if item["mode"] in modes[config["state_mode"]] and (config["include_base"] or item["state_kind"] != "base")]
    if config["unit_or_state_selection"] is not None:
        selected = set(config["unit_or_state_selection"])
        series = [item for item in series if (config["include_base"] and item["state_kind"] == "base") or item["state_id"] in selected or bool(set(item["selected_unit_ids"]).intersection(selected))]
    if not series:
        raise ValueError("No loaded RAPID states match the plotting selection.")
    series = sorted(series, key=lambda item: item["state_kind"] == "base")
    hydrographs = [item["hydro"] for item in series]
    xlim = paper_figure_xlim([data["up_ts"], data["down_ts"]], hydrographs)
    ylim = paper_figure_ylim([data["up_ts"], data["down_ts"]], hydrographs, xlim, config.get("y_margin_fraction", 0.03))
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(data["up_ts"]["time"], data["up_ts"]["discharge"], color="#1f77b4", linewidth=2, label="Upstream gauge")
    ax.plot(data["down_ts"]["time"], data["down_ts"]["discharge"], color="#d62728", linewidth=2, label="Downstream gauge")
    palette = qualitative.Dark24 + qualitative.Alphabet + qualitative.Set3
    for index, item in enumerate(series):
        is_base = item["state_kind"] == "base"
        color = "#111111" if is_base else palette[index % len(palette)]
        ax.plot(item["hydro"]["time_utc"], item["hydro"]["q_outlet_cms"], color=color, linewidth=2, alpha=1.0 if is_base else 0.75, label=item["display_label"])
    ax.set(xlim=xlim, ylim=ylim, xlabel="Time (UTC)", ylabel="Discharge ($m^3/s$)")
    ax.grid(True, color="#d9d9d9", linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_linewidth(1.0); spine.set_color("#333333")
    ax.legend(loc="upper right")
    fig.autofmt_xdate(); fig.tight_layout()
    return fig


def paper_build_matplotlib_figure(data, settings=None):
    """Create a normalized Matplotlib RAPID-overlay figure."""
    config = {**data["config"], **(settings or {})}
    modes = {"independent": {"indep"}, "compound": {"compound"}, "both": {"indep", "compound"}}
    records = data["rapid"]
    if records and "mode" not in records[0]:
        records = [{**item, "mode": row.mode, "state_id": row.state_id, "selected_unit_ids": row.selected_unit_ids} for item, row in zip(records, data["catalog"].itertuples(index=False))]
    series = [item for item in records if item["mode"] in modes[config["state_mode"]] and (config["include_base"] or item["state_kind"] != "base")]
    if config["unit_or_state_selection"] is not None:
        selected = set(config["unit_or_state_selection"])
        series = [item for item in series if (config["include_base"] and item["state_kind"] == "base") or item["state_id"] in selected or bool(set(item["selected_unit_ids"]).intersection(selected))]
    if not series:
        raise ValueError("No loaded RAPID states match the plotting selection.")

    def rising_limb_start(frame, time_column, value_column):
        """Locate the first upward deviation after the selected stable plateau."""
        frame = frame.reset_index(drop=True)
        peak_position = frame[value_column].to_numpy().argmax()
        pre_peak = frame.iloc[:peak_position + 1].copy()
        stable = plateau_mask(
            pre_peak[time_column],
            pre_peak[value_column],
            window=config.get("plateau_window", 40),
            max_rel_range=config.get("plateau_max_rel_range", 0.005),
            max_rel_slope=config.get("plateau_max_rel_slope", 1e-4),
        )
        change_points = np.flatnonzero(np.diff(np.r_[False, stable, False]))
        segments = list(zip(change_points[::2], change_points[1::2] - 1))
        if not segments:
            return pre_peak[time_column].iloc[0]
        # The longest pre-peak stable interval is the event's baseflow plateau.
        plateau_start, plateau_end = max(segments, key=lambda segment: (segment[1] - segment[0], segment[0]))
        plateau = pre_peak.iloc[plateau_start : plateau_end + 1]
        plateau_level = plateau[value_column].median()
        deviation = max(
            config.get("initial_plateau_departure_fraction", 0.01) * abs(plateau_level),
            plateau[value_column].std(ddof=0) * 3,
        )
        after_plateau = pre_peak.iloc[plateau_end + 1 :]
        upward_departure = after_plateau.loc[after_plateau[value_column] > plateau_level + deviation]
        return upward_departure[time_column].iloc[0] if not upward_departure.empty else pre_peak[time_column].iloc[plateau_end]

    def relative_hours(times, origin):
        return (pd.to_datetime(times) - origin).dt.total_seconds() / 3600

    up_start = rising_limb_start(data["up_ts"], "time", "discharge")
    # The downstream record starts by receding; anchor it at its first pre-peak minimum.
    down_pre_peak = data["down_ts"].iloc[: data["down_ts"]["discharge"].to_numpy().argmax() + 1]
    down_minimum = down_pre_peak["discharge"].min()
    down_start = down_pre_peak.loc[down_pre_peak["discharge"].eq(down_minimum), "time"].iloc[0]
    series = sorted(series, key=lambda item: item["state_kind"] == "base")
    rapid_starts = [rising_limb_start(item["hydro"], "time_utc", "q_outlet_cms") for item in series]

    up_x   = relative_hours(data["up_ts"]["time"], up_start)
    down_x = relative_hours(data["down_ts"]["time"], down_start)
    up_start_value = data["up_ts"].loc[(data["up_ts"]["time"] - up_start).abs().idxmin(), "discharge"]
    down_start_value = data["down_ts"].loc[(data["down_ts"]["time"] - down_start).abs().idxmin(), "discharge"]
    downstream_offset = down_start_value - up_start_value
    up_y   = data["up_ts"]["discharge"]
    down_y = data["down_ts"]["discharge"] - downstream_offset

    rapid_xy = [
        (relative_hours(item["hydro"]["time_utc"], start), item["hydro"]["q_outlet_cms"])
        for item, start in zip(series, rapid_starts)
    ]
    # After the upstream record finishes, stop at the last selected RAPID terminal plateau.
    upstream_end_x = float(up_x.max())
    terminal_starts = [
        float(relative_hours(pd.Series([paper_terminal_plateau_start(item["hydro"], "time_utc", "q_outlet_cms", relative_tolerance=config.get("terminal_plateau_relative_tolerance", 0.005))]), start).iloc[0])
        for item, start in zip(series, rapid_starts)
    ]
    eligible_terminal_starts = [value for value in terminal_starts if value >= upstream_end_x]
    x_end = max(eligible_terminal_starts or terminal_starts)
    xlim = (0, x_end)

    # Exclude the hidden RAPID spin-up (x < 0) and post-plateau tail from the y limits.
    visible_values = []
    for x, y in [(up_x, up_y), (down_x, down_y), *rapid_xy]:
        visible_values.extend(pd.Series(y).loc[(x >= 0) & (x <= x_end)].tolist())
    minimum, maximum = min(visible_values), max(visible_values)
    padding = max((maximum - minimum) * config.get("y_margin_fraction", 0.03), 1e-6)
    ylim = (minimum - padding, maximum + padding)

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(up_x, up_y, color="#1f77b4", linewidth=2, linestyle="--", label="Upstream gauge")
    ax.plot(down_x, down_y, color="#d62728", linewidth=2, linestyle="--", label="Downstream gauge")
    palette = qualitative.Dark24 + qualitative.Alphabet + qualitative.Set3
    for index, (item, (x, y)) in enumerate(zip(series, rapid_xy)):
        is_base = item["state_kind"] == "base"
        color = "#111111" if is_base else palette[index % len(palette)]
        ax.plot(x, y, color=color, linewidth=2, alpha=1.0 if is_base else 0.75, label=item["display_label"])
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel("Hours since departure from stable plateau")
    ax.set_ylabel("Discharge ($m^3/s$)")
    ax.grid(True, color="#d9d9d9", linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_linewidth(1.0); spine.set_color("#333333")
    ax.legend(loc="upper right")
    fig.tight_layout()
    return fig


In [ ]:
# `plateau_mask` is defined in the Paper Results functions cell above.
# No separate definition is needed here.


In [ ]:
# Paper Results figure call — edit these settings, then run the stand-alone builder cell below.
PAPER_FIGURE_SETTINGS = {
    "example_id": 21,
    "variant_kind": "anchored",  # "anchored" or "realized"
    "state_mode": "both",  # "independent", "compound", or "both"
    "include_base": True,
    "unit_or_state_selection": [10],  # None for all; e.g. [5, 6] or ["S004_unit_5"]
    "show_vertical_peak_lines": False,
    "y_margin_fraction": 0.03,
    "initial_plateau_departure_fraction": 0.01,  # Matplotlib onset: 1% upward departure from the stable plateau
    "plateau_window": 40,
    "plateau_max_rel_range": 0.005,
    "plateau_max_rel_slope": 1e-5,
    "terminal_plateau_relative_tolerance": 0.001,  # 0.5% tolerance for small terminal-run wriggles
}


In [ ]:
# Data-retrieval call — run after the functions cell and settings cell.
paper_data = paper_retrieve_data(PAPER_FIGURE_SETTINGS)


In [ ]:
# Plotting call — run after data retrieval.
# paper_results_figures = paper_build_figures(paper_data, PAPER_FIGURE_SETTINGS)
# paper_results_figures["observed_gauges"].show(); paper_results_figures["rapid_overlay"].show()


In [ ]:
# matplotlib_simple_figure = paper_build_matplotlib_simple_figure(paper_data, PAPER_FIGURE_SETTINGS)
matplotlib_figure = paper_build_matplotlib_figure(paper_data, PAPER_FIGURE_SETTINGS); plt.show()
